# HydroSeason AOI Rainfall Fetch

This notebook shows how to fetch monthly rainfall for an area of interest and feed it directly into HydroSeason. The recommended path is `source="auto"`: SILO remains the Australian default, CHIRPS v3 monthly rainfall is the global default, and ERA5 is used only when explicitly selected or provided as backup.


Supported AOI vector inputs include GeoJSON, SHP, KML, KMZ, GPKG, GPCK, and other formats readable by GeoPandas. Use a cache directory for repeated runs; HydroSeason stores final monthly tables as Parquet and source metadata as JSON.


In [ ]:
from pathlib import Path

from hydroseason import (
    classify_rainfall,
    generate_html_report,
    get_monthly_aoi_rainfall,
    get_monthly_era5_rainfall,
    get_monthly_silo_rainfall,
    load_vector,
)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

OUTPUT = ROOT / "output"
OUTPUT.mkdir(exist_ok=True)

AOI = ROOT / "data" / "fitzroy_catchment.geojson"
AOI


## Load the AOI

`load_vector()` normalises vector loading for all AOI fetchers. Substitute your own `.shp`, `.kml`, `.kmz`, `.gpkg`, or `.gpck` path here.


In [ ]:
gdf = load_vector(AOI)
gdf.to_crs("EPSG:4326").total_bounds

## Auto AOI Rainfall (Recommended)

`get_monthly_aoi_rainfall(..., source="auto")` chooses the practical default for the polygon: SILO for Australian AOIs and CHIRPS v3 monthly rainfall elsewhere. If you provide `era5_zarr_path`, ERA5 can fill years or months CHIRPS cannot cover.


In [ ]:
RUN_AUTO_FETCH = True
ERA5_ZARR = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

if RUN_AUTO_FETCH:
    auto_monthly = get_monthly_aoi_rainfall(
        gdf,
        start_year=1985,
        end_year=2023,
        source="auto",
        era5_zarr_path=ERA5_ZARR,
        cache_dir=ROOT / "data" / "fetch_cache",
    )
    auto_monthly.to_csv(OUTPUT / "auto_monthly_rainfall.csv", index=False)
    display(auto_monthly.head())
    display(auto_monthly[["Data_Source", "Data_Product"]].drop_duplicates())
else:
    print("Set RUN_AUTO_FETCH = True to fetch monthly AOI rainfall.")


## Explicit Source Options

Use these only when you want to force a specific product. SILO is the Australian gridded product. CHIRPS is available through the AOI wrapper with `source="chirps"`. ERA5 remains the exact/global hourly source, but it is slower because hourly data must be aggregated to monthly rainfall.


In [ ]:
RUN_SILO_FETCH = False
RUN_CHIRPS_FETCH = False
RUN_ERA5_EXACT_FETCH = False

if RUN_SILO_FETCH:
    silo_monthly = get_monthly_silo_rainfall(
        gdf,
        start_year=1985,
        end_year=2023,
        cache_dir=ROOT / "data" / "silo_cache",
    )
    silo_monthly.to_csv(OUTPUT / "silo_monthly_rainfall.csv", index=False)
    display(silo_monthly.head())

if RUN_CHIRPS_FETCH:
    chirps_monthly = get_monthly_aoi_rainfall(
        gdf,
        start_year=1985,
        end_year=2023,
        source="chirps",
        era5_zarr_path=ERA5_ZARR,
        cache_dir=ROOT / "data" / "fetch_cache",
    )
    chirps_monthly.to_csv(OUTPUT / "chirps_monthly_rainfall.csv", index=False)
    display(chirps_monthly.head())
    display(chirps_monthly[["Data_Source", "Data_Product"]].drop_duplicates())

if RUN_ERA5_EXACT_FETCH:
    era5_monthly = get_monthly_era5_rainfall(
        path=ERA5_ZARR,
        gdf=gdf,
        start_year=1985,
        end_year=2023,
        variable="rainfall",
        cache_dir=ROOT / "data" / "era5_cache",
    )
    era5_monthly.to_csv(OUTPUT / "era5_monthly_rainfall.csv", index=False)
    display(era5_monthly.head())

if not any([RUN_SILO_FETCH, RUN_CHIRPS_FETCH, RUN_ERA5_EXACT_FETCH]):
    print("Optional source-specific fetches are disabled.")


## Run HydroSeason After Fetch

Once you have `auto_monthly`, `chirps_monthly`, `silo_monthly`, or `era5_monthly`, the analysis step is identical to a local CSV workflow. The fetch metadata columns are retained for auditability.


In [ ]:
# Example after a fetch has run:
# artifacts = classify_rainfall(auto_monthly)
# result = artifacts.result
# report_path = generate_html_report(artifacts, OUTPUT / "aoi_fetch_report.html")
# result[["Date", "Rainfall_mm", "SeasonType", "Hydro_Year", "Data_Source"]].head()


## CLI Equivalents

```bash
hydroseason fetch --source auto --path gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3 --vector data/fitzroy_catchment.geojson --start-year 1985 --end-year 2023 --cache-dir data/fetch_cache --output output/auto_monthly_rainfall.csv

hydroseason fetch --source chirps --path gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3 --vector data/global_catchment.geojson --start-year 1985 --end-year 2023 --cache-dir data/fetch_cache --output output/chirps_monthly_rainfall.csv

hydroseason fetch --source era5 --path gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3 --vector data/global_catchment.geojson --start-year 1985 --end-year 2023 --variable rainfall --cache-dir data/era5_cache --output output/era5_monthly_rainfall.csv
```


## Fetch-Enabled YAML

A config can fetch and run the pipeline in one command. When `fetch.enabled` is true, `input.csv_path` is optional.

```yaml
output:
  output_csv: output/auto_hydroseason_results.csv

fetch:
  enabled: true
  source: auto
  era5_zarr_path: gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3
  vector_path: data/fitzroy_catchment.geojson
  start_year: 1985
  end_year: 2023
  cache_dir: data/fetch_cache
  era5_fallback: true
```
